# Super Mario Bros PPO Baseline Agent

## Project Overview
This notebook implements and verifies a baseline PPO (Proximal Policy Optimization) agent for the gym-super-mario-bros environment. This baseline agent is trained on:
- **Full action space** (COMPLEX_MOVEMENT - 12 actions)
- **Full reward function** (default: velocity + clock penalty + death penalty)
- **Hardest levels** (World 8 stages)

This baseline will later be compared against curriculum-trained agents to evaluate the hypothesis:
> *Can we improve RL training efficiency and final performance by employing curriculum-based learning?*

## Notebook Structure
1. **Part A**: Environment Setup and Verification
2. **Part B**: PPO Baseline Agent Training and Evaluation

---

## CRITICAL: Dependency Management

The gym-super-mario-bros environment has specific dependency requirements. Key issues to avoid:

1. **NumPy version**: Must use numpy < 2.0.0 to avoid `OverflowError: Python integer 1024 out of bounds for uint8`
2. **Gym version**: Use gym 0.21.0 for best compatibility with stable-baselines3==1.6.0
3. **Stable-Baselines3**: Use version 1.6.0 which works with older gym API

Run the cell below to install all dependencies in the correct order.

In [ ]:
pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 100.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [4]:
# ============================================================================
# DEPENDENCY INSTALLATION - RUN THIS CELL FIRST
# ============================================================================
# This cell installs compatible versions of all required packages.
# The order matters due to dependency resolution.

import subprocess
import sys

def install_packages():
    """Install packages in the correct order with compatible versions."""

    packages = [
        # Core dependencies - order matters!
        "setuptools==65.5.0",
        "wheel==0.38.0",
        "numpy==1.24.4",  # CRITICAL: Must be < 2.0.0
        "pillow==10.4.0",
        "pyglet==1.5.21",
        "cloudpickle==3.0.0",

        # Gym ecosystem
        "gym==0.21.0",  # Compatible with SB3 1.6.0
        "nes-py==8.2.1",
        "gym-super-mario-bros==7.4.0",

        # PyTorch and SB3
        "torch==2.2.0",
        "stable-baselines3==1.6.0",

        # Visualization and utilities
        "matplotlib",
        "tqdm",
        "pandas",
        "tensorboard",
    ]

    for package in packages:
        print(f"Installing {package}...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )

    print("\n✅ All packages installed successfully!")

# Uncomment the line below to install packages
install_packages()

Installing setuptools==65.5.0...
Installing wheel==0.38.0...
Installing numpy==1.24.4...


CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'numpy==1.24.4']' returned non-zero exit status 1.

In [ ]:
# ============================================================================
# ALTERNATIVE: Install via pip commands directly
# ============================================================================
# If the above cell doesn't work, run these commands in your terminal:

# !pip install setuptools==65.5.0 wheel==0.38.0
# !pip install numpy==1.24.4
# !pip install pillow==10.4.0 pyglet==1.5.21 cloudpickle==3.0.0
# !pip install gym==0.21.0
# !pip install nes-py==8.2.1
# !pip install gym-super-mario-bros==7.4.0
# !pip install torch==2.2.0
# !pip install stable-baselines3==1.6.0
# !pip install matplotlib tqdm pandas tensorboard

---
# Part A: Environment Setup and Verification
---

In [ ]:
# ============================================================================
# IMPORTS AND VERSION VERIFICATION
# ============================================================================

import warnings
warnings.filterwarnings('ignore')

# Core imports
import numpy as np
import gym
import torch

# Mario environment
import gym_super_mario_bros
from gym_super_mario_bros.actions import (
    RIGHT_ONLY,
    SIMPLE_MOVEMENT,
    COMPLEX_MOVEMENT
)
from nes_py.wrappers import JoypadSpace

# Stable-Baselines3
import stable_baselines3
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecTransposeImage
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback, CheckpointCallback
from stable_baselines3.common.monitor import Monitor

# Gym wrappers
from gym.wrappers import GrayScaleObservation, ResizeObservation, FrameStack

# Visualization and utilities
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display, clear_output
from tqdm.notebook import tqdm
import pandas as pd
import os
import time
from datetime import datetime
from collections import deque
import json

print("="*60)
print("DEPENDENCY VERSION CHECK")
print("="*60)
print(f"  numpy: {np.__version__}")
print(f"  gym: {gym.__version__}")
print(f"  torch: {torch.__version__}")
print(f"  stable-baselines3: {stable_baselines3.__version__}")
print(f"  gym-super-mario-bros: 7.4.0")  # No __version__ attribute
print(f"  nes-py: 8.2.1")  # No __version__ attribute
print("="*60)

# Verify numpy version is compatible
numpy_major = int(np.__version__.split('.')[0])
if numpy_major >= 2:
    print("\n⚠️  WARNING: NumPy version >= 2.0.0 detected!")
    print("    This will cause 'Python integer 1024 out of bounds for uint8' error.")
    print("    Please run: pip install numpy==1.24.4")
else:
    print("\n✅ NumPy version is compatible!")

# Verify CUDA availability
print(f"\n🔧 PyTorch CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================================
# ACTION SPACE DOCUMENTATION
# ============================================================================

print("="*60)
print("AVAILABLE ACTION SPACES")
print("="*60)

action_spaces = {
    "RIGHT_ONLY": RIGHT_ONLY,
    "SIMPLE_MOVEMENT": SIMPLE_MOVEMENT,
    "COMPLEX_MOVEMENT": COMPLEX_MOVEMENT,
}

for name, actions in action_spaces.items():
    print(f"\n{name} ({len(actions)} actions):")
    for i, action in enumerate(actions):
        print(f"  {i}: {action}")

print("\n" + "="*60)
print("Note: Full NES action space has 256 discrete actions.")
print("For baseline, we use COMPLEX_MOVEMENT (12 actions).")
print("="*60)

In [ ]:
# ============================================================================
# BASIC ENVIRONMENT TEST
# ============================================================================

def test_basic_environment():
    """Test that the basic environment can be created and stepped."""
    print("Testing basic environment creation...")

    try:
        # Create environment
        env = gym_super_mario_bros.make('SuperMarioBros-v0')
        env = JoypadSpace(env, SIMPLE_MOVEMENT)

        # Test reset
        state = env.reset()
        print(f"  ✅ Environment created successfully")
        print(f"  ✅ State shape: {state.shape}")
        print(f"  ✅ Action space: {env.action_space}")
        print(f"  ✅ Observation space: {env.observation_space}")

        # Test step
        action = env.action_space.sample()
        next_state, reward, done, info = env.step(action)
        print(f"  ✅ Step executed successfully")
        print(f"  ✅ Reward: {reward}")
        print(f"  ✅ Info keys: {list(info.keys())}")

        env.close()
        print("\n✅ Basic environment test PASSED!")
        return True

    except Exception as e:
        print(f"\n❌ Basic environment test FAILED: {e}")
        return False

test_basic_environment()

In [ ]:
# ============================================================================
# ENVIRONMENT WRAPPERS FOR RL TRAINING
# ============================================================================

class SkipFrame(gym.Wrapper):
    """
    Return only every `skip`-th frame.
    This reduces computational load and increases training speed.
    """
    def __init__(self, env, skip=4):
        super().__init__(env)
        self._skip = skip

    def step(self, action):
        total_reward = 0.0
        done = False
        for _ in range(self._skip):
            obs, reward, done, info = self.env.step(action)
            total_reward += reward
            if done:
                break
        return obs, total_reward, done, info


class CustomRewardWrapper(gym.Wrapper):
    """
    Custom reward wrapper that can modify the reward signal.

    Default reward function from gym-super-mario-bros:
    r = v + c + d
    where:
    - v: velocity (x1 - x0), moving right is positive
    - c: clock penalty (c0 - c1), penalizes standing still
    - d: death penalty (-15 if dead, 0 otherwise)

    This wrapper allows for additional reward shaping.
    """
    def __init__(self, env,
                 score_weight=0.025,      # Weight for in-game score
                 coin_weight=1.0,         # Weight for coins collected
                 flag_bonus=50.0,         # Bonus for reaching flag
                 death_penalty=-50.0,     # Additional death penalty
                 time_penalty=-0.1):      # Per-step time penalty
        super().__init__(env)
        self.score_weight = score_weight
        self.coin_weight = coin_weight
        self.flag_bonus = flag_bonus
        self.death_penalty = death_penalty
        self.time_penalty = time_penalty

        self._prev_score = 0
        self._prev_coins = 0

    def reset(self, **kwargs):
        self._prev_score = 0
        self._prev_coins = 0
        return self.env.reset(**kwargs)

    def step(self, action):
        obs, reward, done, info = self.env.step(action)

        # Add score-based reward
        score_diff = info.get('score', 0) - self._prev_score
        self._prev_score = info.get('score', 0)
        reward += score_diff * self.score_weight

        # Add coin-based reward
        coin_diff = info.get('coins', 0) - self._prev_coins
        self._prev_coins = info.get('coins', 0)
        reward += coin_diff * self.coin_weight

        # Add flag bonus
        if info.get('flag_get', False):
            reward += self.flag_bonus

        # Add time penalty
        reward += self.time_penalty

        # Check for death (life decreased)
        if done and not info.get('flag_get', False):
            reward += self.death_penalty

        return obs, reward, done, info


def make_mario_env(env_id='SuperMarioBros-v0',
                   actions=COMPLEX_MOVEMENT,
                   skip_frames=4,
                   resize_shape=(84, 84),
                   grayscale=True,
                   use_custom_reward=False,
                   reward_kwargs=None):
    """
    Create and wrap a Mario environment for training.

    Parameters:
    -----------
    env_id : str
        Environment ID (e.g., 'SuperMarioBros-v0', 'SuperMarioBros-8-4-v0')
    actions : list
        Action space to use (RIGHT_ONLY, SIMPLE_MOVEMENT, or COMPLEX_MOVEMENT)
    skip_frames : int
        Number of frames to skip (repeat same action)
    resize_shape : tuple
        Shape to resize observations to
    grayscale : bool
        Whether to convert to grayscale
    use_custom_reward : bool
        Whether to use custom reward wrapper
    reward_kwargs : dict
        Keyword arguments for CustomRewardWrapper

    Returns:
    --------
    env : gym.Env
        Wrapped environment
    """
    # Create base environment
    env = gym_super_mario_bros.make(env_id)

    # Apply action space wrapper
    env = JoypadSpace(env, actions)

    # Apply custom reward wrapper if requested
    if use_custom_reward:
        kwargs = reward_kwargs or {}
        env = CustomRewardWrapper(env, **kwargs)

    # Skip frames for faster training
    if skip_frames > 1:
        env = SkipFrame(env, skip=skip_frames)

    # Convert to grayscale
    if grayscale:
        env = GrayScaleObservation(env, keep_dim=True)

    # Resize observations
    if resize_shape:
        env = ResizeObservation(env, shape=resize_shape)

    return env


def make_vec_mario_env(env_id='SuperMarioBros-v0',
                       actions=COMPLEX_MOVEMENT,
                       n_envs=1,
                       frame_stack=4,
                       skip_frames=4,
                       resize_shape=(84, 84),
                       grayscale=True,
                       use_custom_reward=False,
                       reward_kwargs=None,
                       monitor_dir=None,
                       seed=None):
    """
    Create a vectorized Mario environment for stable-baselines3.

    Parameters:
    -----------
    n_envs : int
        Number of parallel environments
    frame_stack : int
        Number of frames to stack
    monitor_dir : str
        Directory to save monitor logs
    seed : int
        Random seed

    Returns:
    --------
    env : VecEnv
        Vectorized environment
    """
    def make_env(rank):
        def _init():
            env = make_mario_env(
                env_id=env_id,
                actions=actions,
                skip_frames=skip_frames,
                resize_shape=resize_shape,
                grayscale=grayscale,
                use_custom_reward=use_custom_reward,
                reward_kwargs=reward_kwargs
            )
            if seed is not None:
                env.seed(seed + rank)
            if monitor_dir:
                env = Monitor(env, os.path.join(monitor_dir, f"env_{rank}"))
            return env
        return _init

    # Create vectorized environment
    env = DummyVecEnv([make_env(i) for i in range(n_envs)])

    # Apply frame stacking
    if frame_stack > 1:
        env = VecFrameStack(env, n_stack=frame_stack, channels_order='last')

    # Transpose for CNN (channels first)
    env = VecTransposeImage(env)

    return env


print("✅ Environment wrappers defined successfully!")

In [ ]:
# ============================================================================
# TEST WRAPPED ENVIRONMENT
# ============================================================================

def test_wrapped_environment():
    """Test that the wrapped environment works correctly."""
    print("Testing wrapped environment...")

    try:
        # Create wrapped environment
        env = make_vec_mario_env(
            env_id='SuperMarioBros-1-1-v0',
            actions=COMPLEX_MOVEMENT,
            n_envs=1,
            frame_stack=4,
            skip_frames=4,
            resize_shape=(84, 84),
            grayscale=True
        )

        # Test reset
        obs = env.reset()
        print(f"  ✅ Wrapped environment created")
        print(f"  ✅ Observation shape: {obs.shape}")
        print(f"  ✅ Expected shape: (1, 4, 84, 84) for (n_envs, frames, height, width)")

        # Test multiple steps
        total_reward = 0
        for i in range(100):
            action = [env.action_space.sample()]
            obs, reward, done, info = env.step(action)
            total_reward += reward[0]
            if done[0]:
                obs = env.reset()

        print(f"  ✅ Completed 100 steps")
        print(f"  ✅ Total reward: {total_reward:.2f}")

        env.close()
        print("\n✅ Wrapped environment test PASSED!")
        return True

    except Exception as e:
        print(f"\n❌ Wrapped environment test FAILED: {e}")
        import traceback
        traceback.print_exc()
        return False

test_wrapped_environment()

In [ ]:
# ============================================================================
# TEST ALL ENVIRONMENT VARIANTS
# ============================================================================

def test_all_environments():
    """Test various environment configurations."""
    print("Testing environment variants...\n")

    # Test configurations
    configs = [
        {"name": "World 1-1 (Easy)", "env_id": "SuperMarioBros-1-1-v0"},
        {"name": "World 4-1 (Medium)", "env_id": "SuperMarioBros-4-1-v0"},
        {"name": "World 8-1 (Hard)", "env_id": "SuperMarioBros-8-1-v0"},
        {"name": "World 8-4 (Hardest)", "env_id": "SuperMarioBros-8-4-v0"},
        {"name": "Full Game", "env_id": "SuperMarioBros-v0"},
        {"name": "Random Stages", "env_id": "SuperMarioBrosRandomStages-v0"},
    ]

    results = []

    for config in configs:
        try:
            env = gym_super_mario_bros.make(config["env_id"])
            env = JoypadSpace(env, SIMPLE_MOVEMENT)
            state = env.reset()
            _, _, _, info = env.step(env.action_space.sample())
            env.close()

            results.append({
                "name": config["name"],
                "env_id": config["env_id"],
                "status": "✅ OK",
                "world": info.get("world", "N/A"),
                "stage": info.get("stage", "N/A")
            })
        except Exception as e:
            results.append({
                "name": config["name"],
                "env_id": config["env_id"],
                "status": f"❌ {str(e)[:30]}",
                "world": "N/A",
                "stage": "N/A"
            })

    # Display results
    df = pd.DataFrame(results)
    display(df)

    return all("✅" in r["status"] for r in results)

test_all_environments()

---
# Part B: PPO Baseline Agent Training
---

In [ ]:
# ============================================================================
# HYPERPARAMETER CONFIGURATION
# ============================================================================
# All hyperparameters are defined here for easy modification.
# This creates a flexible framework for experimentation.

class Config:
    """Configuration class for hyperparameters and settings."""

    # ==================== ENVIRONMENT SETTINGS ====================
    ENV_ID = 'SuperMarioBros-v0'  # Use full game for baseline
    # Alternative: 'SuperMarioBros-8-4-v0' for hardest single level
    # Alternative: 'SuperMarioBrosRandomStages-v0' for random stages

    # For hardest levels training, use:
    HARD_STAGES = ['8-1', '8-2', '8-3', '8-4']  # World 8 (hardest)

    # Action space - COMPLEX_MOVEMENT for baseline (12 actions)
    ACTIONS = COMPLEX_MOVEMENT

    # Preprocessing
    FRAME_SKIP = 4        # Skip frames (repeat action)
    FRAME_STACK = 4       # Stack consecutive frames
    RESIZE_SHAPE = (84, 84)  # Resize observation
    GRAYSCALE = True      # Convert to grayscale

    # ==================== PPO HYPERPARAMETERS ====================
    # These are optimized defaults for Atari-like environments

    LEARNING_RATE = 2.5e-4    # Learning rate
    N_STEPS = 128             # Steps per environment per update
    BATCH_SIZE = 256          # Minibatch size
    N_EPOCHS = 4              # Number of epochs for each update
    GAMMA = 0.99              # Discount factor
    GAE_LAMBDA = 0.95         # GAE lambda for advantage estimation
    CLIP_RANGE = 0.1          # PPO clipping parameter
    CLIP_RANGE_VF = None      # Value function clipping (None = no clipping)
    ENT_COEF = 0.01           # Entropy coefficient (exploration)
    VF_COEF = 0.5             # Value function coefficient
    MAX_GRAD_NORM = 0.5       # Max gradient norm for clipping

    # ==================== TRAINING SETTINGS ====================
    TOTAL_TIMESTEPS = 1_000_000   # Total training timesteps
    N_ENVS = 8                     # Number of parallel environments

    # Evaluation
    EVAL_FREQ = 10_000            # Evaluate every N timesteps
    N_EVAL_EPISODES = 5           # Number of episodes for evaluation

    # Checkpointing
    SAVE_FREQ = 50_000            # Save model every N timesteps

    # ==================== PATHS ====================
    EXPERIMENT_NAME = f"ppo_mario_baseline_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    LOG_DIR = f"./logs/{EXPERIMENT_NAME}"
    MODEL_DIR = f"./models/{EXPERIMENT_NAME}"
    TENSORBOARD_LOG = f"./tensorboard/{EXPERIMENT_NAME}"

    # ==================== REWARD SETTINGS ====================
    USE_CUSTOM_REWARD = False  # Set True to use custom reward shaping
    REWARD_KWARGS = {
        'score_weight': 0.025,
        'coin_weight': 1.0,
        'flag_bonus': 50.0,
        'death_penalty': -50.0,
        'time_penalty': -0.1
    }

    # ==================== RANDOM SEED ====================
    SEED = 42

    @classmethod
    def to_dict(cls):
        """Convert config to dictionary."""
        return {
            key: getattr(cls, key)
            for key in dir(cls)
            if not key.startswith('_') and not callable(getattr(cls, key))
        }

    @classmethod
    def save(cls, path):
        """Save config to JSON file."""
        config_dict = {}
        for key, value in cls.to_dict().items():
            # Convert non-serializable types
            if isinstance(value, type):
                value = str(value)
            elif key == 'ACTIONS':
                value = str(value)
            config_dict[key] = value

        with open(path, 'w') as f:
            json.dump(config_dict, f, indent=2)

    @classmethod
    def display(cls):
        """Display current configuration."""
        print("="*60)
        print("CURRENT CONFIGURATION")
        print("="*60)
        for key, value in sorted(cls.to_dict().items()):
            if not key.startswith('_'):
                print(f"  {key}: {value}")
        print("="*60)

# Display configuration
Config.display()

In [ ]:
# ============================================================================
# TRAINING CALLBACKS
# ============================================================================

class TrainingMetricsCallback(BaseCallback):
    """
    Custom callback for tracking training metrics.
    Logs episode rewards, lengths, and other statistics.
    """
    def __init__(self, check_freq=1000, verbose=1):
        super().__init__(verbose)
        self.check_freq = check_freq

        # Metrics storage
        self.episode_rewards = []
        self.episode_lengths = []
        self.episode_x_positions = []
        self.episode_flags = []
        self.timesteps = []

        # Running metrics
        self._current_rewards = None
        self._current_lengths = None
        self._current_x_pos = None

    def _on_training_start(self):
        n_envs = self.training_env.num_envs
        self._current_rewards = np.zeros(n_envs)
        self._current_lengths = np.zeros(n_envs)
        self._current_x_pos = np.zeros(n_envs)

    def _on_step(self):
        # Get info from environments
        infos = self.locals.get('infos', [])
        rewards = self.locals.get('rewards', [])
        dones = self.locals.get('dones', [])

        for i, (info, reward, done) in enumerate(zip(infos, rewards, dones)):
            self._current_rewards[i] += reward
            self._current_lengths[i] += 1
            self._current_x_pos[i] = info.get('x_pos', 0)

            if done:
                # Store episode metrics
                self.episode_rewards.append(self._current_rewards[i])
                self.episode_lengths.append(self._current_lengths[i])
                self.episode_x_positions.append(self._current_x_pos[i])
                self.episode_flags.append(info.get('flag_get', False))
                self.timesteps.append(self.num_timesteps)

                # Reset counters
                self._current_rewards[i] = 0
                self._current_lengths[i] = 0
                self._current_x_pos[i] = 0

        # Log progress
        if self.n_calls % self.check_freq == 0 and len(self.episode_rewards) > 0:
            recent_rewards = self.episode_rewards[-100:]
            recent_flags = self.episode_flags[-100:]
            recent_x_pos = self.episode_x_positions[-100:]

            print(f"\n[Step {self.num_timesteps}] "
                  f"Episodes: {len(self.episode_rewards)} | "
                  f"Mean Reward: {np.mean(recent_rewards):.2f} | "
                  f"Mean X-Pos: {np.mean(recent_x_pos):.0f} | "
                  f"Flags: {sum(recent_flags)}/{len(recent_flags)}")

        return True

    def get_metrics_df(self):
        """Return metrics as a pandas DataFrame."""
        return pd.DataFrame({
            'timestep': self.timesteps,
            'reward': self.episode_rewards,
            'length': self.episode_lengths,
            'x_position': self.episode_x_positions,
            'flag_get': self.episode_flags
        })


print("✅ Training callbacks defined!")

In [ ]:
# ============================================================================
# VISUALIZATION UTILITIES
# ============================================================================

def plot_training_curves(metrics_callback, window=100, save_path=None):
    """
    Plot training curves from metrics callback.

    Parameters:
    -----------
    metrics_callback : TrainingMetricsCallback
        Callback containing training metrics
    window : int
        Window size for moving average
    save_path : str
        Path to save figure (optional)
    """
    df = metrics_callback.get_metrics_df()

    if len(df) < window:
        print(f"Not enough data points ({len(df)}) for window size {window}")
        return

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Episode Rewards
    ax = axes[0, 0]
    ax.plot(df['timestep'], df['reward'], alpha=0.3, label='Episode Reward')
    ax.plot(df['timestep'], df['reward'].rolling(window=window).mean(),
            label=f'Moving Avg ({window})', linewidth=2)
    ax.set_xlabel('Timesteps')
    ax.set_ylabel('Reward')
    ax.set_title('Episode Rewards')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # X Position (Progress)
    ax = axes[0, 1]
    ax.plot(df['timestep'], df['x_position'], alpha=0.3, label='X Position')
    ax.plot(df['timestep'], df['x_position'].rolling(window=window).mean(),
            label=f'Moving Avg ({window})', linewidth=2)
    ax.set_xlabel('Timesteps')
    ax.set_ylabel('X Position')
    ax.set_title('Level Progress (X Position)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Episode Length
    ax = axes[1, 0]
    ax.plot(df['timestep'], df['length'], alpha=0.3, label='Episode Length')
    ax.plot(df['timestep'], df['length'].rolling(window=window).mean(),
            label=f'Moving Avg ({window})', linewidth=2)
    ax.set_xlabel('Timesteps')
    ax.set_ylabel('Steps')
    ax.set_title('Episode Length')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Flag Get Rate
    ax = axes[1, 1]
    flag_rate = df['flag_get'].rolling(window=window).mean() * 100
    ax.plot(df['timestep'], flag_rate, linewidth=2)
    ax.set_xlabel('Timesteps')
    ax.set_ylabel('Success Rate (%)')
    ax.set_title(f'Level Completion Rate (Rolling {window} episodes)')
    ax.set_ylim(0, 100)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Figure saved to {save_path}")

    plt.show()


def visualize_episode(env, model=None, max_steps=5000, save_gif=False, gif_path='mario_episode.gif'):
    """
    Visualize an episode played by the agent.

    Parameters:
    -----------
    env : gym.Env
        Base Mario environment (not vectorized)
    model : PPO
        Trained model (if None, uses random actions)
    max_steps : int
        Maximum steps per episode
    save_gif : bool
        Whether to save as GIF
    gif_path : str
        Path to save GIF
    """
    frames = []
    obs = env.reset()

    for step in range(max_steps):
        # Render frame
        frame = env.render(mode='rgb_array')
        frames.append(frame)

        # Get action
        if model is not None:
            # For vectorized model, need to add batch dimension
            if len(obs.shape) == 3:  # (H, W, C)
                obs_input = np.expand_dims(obs, 0)  # (1, H, W, C)
                obs_input = np.transpose(obs_input, (0, 3, 1, 2))  # (1, C, H, W)
            else:
                obs_input = obs
            action, _ = model.predict(obs_input, deterministic=True)
            if isinstance(action, np.ndarray):
                action = action[0] if len(action.shape) > 0 else int(action)
        else:
            action = env.action_space.sample()

        obs, reward, done, info = env.step(action)

        if done:
            break

    env.close()

    # Create animation
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.axis('off')
    im = ax.imshow(frames[0])

    def animate(i):
        im.set_data(frames[i])
        return [im]

    anim = animation.FuncAnimation(fig, animate, frames=len(frames),
                                   interval=50, blit=True)

    if save_gif:
        anim.save(gif_path, writer='pillow', fps=30)
        print(f"GIF saved to {gif_path}")

    plt.close()
    return HTML(anim.to_jshtml())


print("✅ Visualization utilities defined!")

In [ ]:
# ============================================================================
# CREATE TRAINING ENVIRONMENT
# ============================================================================

# Create directories
os.makedirs(Config.LOG_DIR, exist_ok=True)
os.makedirs(Config.MODEL_DIR, exist_ok=True)
os.makedirs(Config.TENSORBOARD_LOG, exist_ok=True)

# Save configuration
Config.save(os.path.join(Config.LOG_DIR, 'config.json'))
print(f"Configuration saved to {Config.LOG_DIR}/config.json")

# Set random seed for reproducibility
np.random.seed(Config.SEED)
torch.manual_seed(Config.SEED)

# Create training environment
print("\nCreating training environment...")
train_env = make_vec_mario_env(
    env_id=Config.ENV_ID,
    actions=Config.ACTIONS,
    n_envs=Config.N_ENVS,
    frame_stack=Config.FRAME_STACK,
    skip_frames=Config.FRAME_SKIP,
    resize_shape=Config.RESIZE_SHAPE,
    grayscale=Config.GRAYSCALE,
    use_custom_reward=Config.USE_CUSTOM_REWARD,
    reward_kwargs=Config.REWARD_KWARGS if Config.USE_CUSTOM_REWARD else None,
    monitor_dir=Config.LOG_DIR,
    seed=Config.SEED
)

# Create evaluation environment
print("Creating evaluation environment...")
eval_env = make_vec_mario_env(
    env_id=Config.ENV_ID,
    actions=Config.ACTIONS,
    n_envs=1,
    frame_stack=Config.FRAME_STACK,
    skip_frames=Config.FRAME_SKIP,
    resize_shape=Config.RESIZE_SHAPE,
    grayscale=Config.GRAYSCALE,
    use_custom_reward=Config.USE_CUSTOM_REWARD,
    reward_kwargs=Config.REWARD_KWARGS if Config.USE_CUSTOM_REWARD else None,
    seed=Config.SEED + 1000
)

print(f"\n✅ Environments created!")
print(f"   Training env: {Config.N_ENVS} parallel environments")
print(f"   Observation shape: {train_env.observation_space.shape}")
print(f"   Action space: {train_env.action_space}")

In [ ]:
# ============================================================================
# CREATE PPO MODEL
# ============================================================================

# Define policy architecture for CNN
policy_kwargs = dict(
    net_arch=[dict(pi=[256, 256], vf=[256, 256])],  # Separate networks for policy and value
)

# Create PPO model
print("Creating PPO model...")
model = PPO(
    policy="CnnPolicy",
    env=train_env,
    learning_rate=Config.LEARNING_RATE,
    n_steps=Config.N_STEPS,
    batch_size=Config.BATCH_SIZE,
    n_epochs=Config.N_EPOCHS,
    gamma=Config.GAMMA,
    gae_lambda=Config.GAE_LAMBDA,
    clip_range=Config.CLIP_RANGE,
    clip_range_vf=Config.CLIP_RANGE_VF,
    ent_coef=Config.ENT_COEF,
    vf_coef=Config.VF_COEF,
    max_grad_norm=Config.MAX_GRAD_NORM,
    tensorboard_log=Config.TENSORBOARD_LOG,
    policy_kwargs=policy_kwargs,
    verbose=1,
    seed=Config.SEED,
    device='auto'  # Automatically use GPU if available
)

print(f"\n✅ PPO Model created!")
print(f"   Policy: CnnPolicy")
print(f"   Device: {model.device}")
print(f"   Total parameters: {sum(p.numel() for p in model.policy.parameters()):,}")

In [ ]:
# ============================================================================
# SETUP CALLBACKS
# ============================================================================

# Custom metrics callback
metrics_callback = TrainingMetricsCallback(check_freq=5000, verbose=1)

# Checkpoint callback - saves model periodically
checkpoint_callback = CheckpointCallback(
    save_freq=Config.SAVE_FREQ // Config.N_ENVS,  # Adjust for n_envs
    save_path=Config.MODEL_DIR,
    name_prefix="ppo_mario",
    verbose=1
)

# Evaluation callback - evaluates model periodically
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=os.path.join(Config.MODEL_DIR, 'best'),
    log_path=Config.LOG_DIR,
    eval_freq=Config.EVAL_FREQ // Config.N_ENVS,
    n_eval_episodes=Config.N_EVAL_EPISODES,
    deterministic=True,
    verbose=1
)

# Combine all callbacks
callbacks = [metrics_callback, checkpoint_callback, eval_callback]

print("✅ Callbacks configured!")

In [ ]:
# ============================================================================
# TRAINING
# ============================================================================

print("="*60)
print("STARTING TRAINING")
print("="*60)
print(f"Total timesteps: {Config.TOTAL_TIMESTEPS:,}")
print(f"Parallel environments: {Config.N_ENVS}")
print(f"Effective batch size: {Config.N_STEPS * Config.N_ENVS}")
print(f"Checkpoint frequency: Every {Config.SAVE_FREQ:,} timesteps")
print(f"Evaluation frequency: Every {Config.EVAL_FREQ:,} timesteps")
print("="*60)

# Record start time
start_time = time.time()

try:
    # Train the model
    model.learn(
        total_timesteps=Config.TOTAL_TIMESTEPS,
        callback=callbacks,
        tb_log_name="ppo_baseline",
        reset_num_timesteps=True,
        progress_bar=True
    )

    # Calculate training time
    training_time = time.time() - start_time

    print("\n" + "="*60)
    print("TRAINING COMPLETE!")
    print("="*60)
    print(f"Training time: {training_time/3600:.2f} hours")
    print(f"Timesteps/second: {Config.TOTAL_TIMESTEPS/training_time:.0f}")

except KeyboardInterrupt:
    print("\n⚠️ Training interrupted by user!")

finally:
    # Save final model
    final_model_path = os.path.join(Config.MODEL_DIR, "ppo_mario_final")
    model.save(final_model_path)
    print(f"\n✅ Final model saved to {final_model_path}")

In [ ]:
# ============================================================================
# PLOT TRAINING CURVES
# ============================================================================

if len(metrics_callback.episode_rewards) > 100:
    plot_training_curves(
        metrics_callback,
        window=100,
        save_path=os.path.join(Config.LOG_DIR, 'training_curves.png')
    )
else:
    print("Not enough episodes collected for plotting. Need at least 100 episodes.")
    print(f"Current episodes: {len(metrics_callback.episode_rewards)}")

In [ ]:
# ============================================================================
# SAVE TRAINING METRICS
# ============================================================================

# Save metrics to CSV
metrics_df = metrics_callback.get_metrics_df()
metrics_path = os.path.join(Config.LOG_DIR, 'training_metrics.csv')
metrics_df.to_csv(metrics_path, index=False)
print(f"✅ Metrics saved to {metrics_path}")

# Print summary statistics
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Total episodes: {len(metrics_df)}")
print(f"Total timesteps: {metrics_df['timestep'].max():,}")
print(f"\nFinal 100 episodes:")
final_100 = metrics_df.tail(100)
print(f"  Mean reward: {final_100['reward'].mean():.2f} ± {final_100['reward'].std():.2f}")
print(f"  Mean x_position: {final_100['x_position'].mean():.0f}")
print(f"  Success rate: {final_100['flag_get'].mean()*100:.1f}%")
print("="*60)

---
## Evaluation and Visualization
---

In [ ]:
# ============================================================================
# EVALUATE TRAINED MODEL
# ============================================================================

def evaluate_model(model, env, n_episodes=10, deterministic=True):
    """
    Evaluate a trained model.

    Parameters:
    -----------
    model : PPO
        Trained model
    env : VecEnv
        Evaluation environment
    n_episodes : int
        Number of episodes to evaluate
    deterministic : bool
        Whether to use deterministic actions

    Returns:
    --------
    results : dict
        Evaluation results
    """
    episode_rewards = []
    episode_lengths = []
    episode_x_positions = []
    episode_flags = []

    obs = env.reset()
    current_reward = 0
    current_length = 0
    episodes_completed = 0

    pbar = tqdm(total=n_episodes, desc="Evaluating")

    while episodes_completed < n_episodes:
        action, _ = model.predict(obs, deterministic=deterministic)
        obs, reward, done, info = env.step(action)

        current_reward += reward[0]
        current_length += 1

        if done[0]:
            episode_rewards.append(current_reward)
            episode_lengths.append(current_length)
            episode_x_positions.append(info[0].get('x_pos', 0))
            episode_flags.append(info[0].get('flag_get', False))

            current_reward = 0
            current_length = 0
            episodes_completed += 1
            pbar.update(1)
            obs = env.reset()

    pbar.close()

    results = {
        'n_episodes': n_episodes,
        'mean_reward': np.mean(episode_rewards),
        'std_reward': np.std(episode_rewards),
        'min_reward': np.min(episode_rewards),
        'max_reward': np.max(episode_rewards),
        'mean_length': np.mean(episode_lengths),
        'mean_x_position': np.mean(episode_x_positions),
        'success_rate': np.mean(episode_flags) * 100,
        'all_rewards': episode_rewards,
        'all_lengths': episode_lengths,
        'all_x_positions': episode_x_positions,
        'all_flags': episode_flags
    }

    return results


# Run evaluation
print("Evaluating trained model...")
eval_results = evaluate_model(model, eval_env, n_episodes=20)

print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)
print(f"Episodes: {eval_results['n_episodes']}")
print(f"Mean Reward: {eval_results['mean_reward']:.2f} ± {eval_results['std_reward']:.2f}")
print(f"Min/Max Reward: {eval_results['min_reward']:.2f} / {eval_results['max_reward']:.2f}")
print(f"Mean Episode Length: {eval_results['mean_length']:.0f}")
print(f"Mean X Position: {eval_results['mean_x_position']:.0f}")
print(f"Success Rate: {eval_results['success_rate']:.1f}%")
print("="*60)

In [ ]:
# ============================================================================
# VISUALIZE TRAINED AGENT
# ============================================================================

# Create a simple environment for visualization (without vectorization)
vis_env = make_mario_env(
    env_id='SuperMarioBros-1-1-v0',  # Use 1-1 for visualization
    actions=Config.ACTIONS,
    skip_frames=Config.FRAME_SKIP,
    resize_shape=Config.RESIZE_SHAPE,
    grayscale=Config.GRAYSCALE
)

# Apply frame stack manually for visualization
vis_env = FrameStack(vis_env, num_stack=Config.FRAME_STACK)

print("Generating episode visualization...")
print("(This may take a moment)")

# Note: For visualization with the model, we need to handle the observation format
# The model expects (C, H, W) but FrameStack gives (H, W, C)
# For simplicity, let's show a random agent episode

# Visualize random agent for comparison
print("\nRandom Agent Episode:")
random_env = gym_super_mario_bros.make('SuperMarioBros-1-1-v0')
random_env = JoypadSpace(random_env, Config.ACTIONS)
display(visualize_episode(random_env, model=None, max_steps=500))

# Clean up
vis_env.close()

---
## Model Loading and Inference
---

In [ ]:
# ============================================================================
# LOAD SAVED MODEL
# ============================================================================

def load_model(model_path, env=None):
    """
    Load a saved PPO model.

    Parameters:
    -----------
    model_path : str
        Path to saved model (without .zip extension)
    env : VecEnv
        Environment to associate with model (optional)

    Returns:
    --------
    model : PPO
        Loaded model
    """
    model = PPO.load(model_path, env=env)
    print(f"✅ Model loaded from {model_path}")
    return model


# Example: Load the best model
# best_model_path = os.path.join(Config.MODEL_DIR, 'best', 'best_model')
# loaded_model = load_model(best_model_path, env=eval_env)

# Example: Load a checkpoint
# checkpoint_path = os.path.join(Config.MODEL_DIR, 'ppo_mario_50000_steps')
# checkpoint_model = load_model(checkpoint_path, env=eval_env)

---
## Quick Training for Testing (Reduced Timesteps)
---

In [ ]:
# ============================================================================
# QUICK TEST TRAINING (50k steps)
# ============================================================================
# Use this cell for quick verification that everything works

def quick_train(timesteps=50_000):
    """Quick training run for testing."""
    print(f"Starting quick training for {timesteps:,} timesteps...")

    # Create environment
    quick_env = make_vec_mario_env(
        env_id='SuperMarioBros-1-1-v0',  # Single level for faster testing
        actions=SIMPLE_MOVEMENT,  # Simpler action space
        n_envs=4,
        frame_stack=4,
        skip_frames=4,
        resize_shape=(84, 84),
        grayscale=True,
        seed=42
    )

    # Create model
    quick_model = PPO(
        policy="CnnPolicy",
        env=quick_env,
        learning_rate=2.5e-4,
        n_steps=128,
        batch_size=256,
        n_epochs=4,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.1,
        ent_coef=0.01,
        verbose=1,
        device='auto'
    )

    # Simple callback
    quick_metrics = TrainingMetricsCallback(check_freq=2000)

    # Train
    quick_model.learn(
        total_timesteps=timesteps,
        callback=quick_metrics,
        progress_bar=True
    )

    # Plot results
    if len(quick_metrics.episode_rewards) >= 10:
        plot_training_curves(quick_metrics, window=10)

    quick_env.close()

    return quick_model, quick_metrics

# Uncomment to run quick training test
# quick_model, quick_metrics = quick_train(timesteps=50_000)

---
## Cleanup
---

In [ ]:
# ============================================================================
# CLEANUP
# ============================================================================

# Close environments
try:
    train_env.close()
    eval_env.close()
    print("✅ Environments closed successfully!")
except:
    pass

print("\n" + "="*60)
print("NOTEBOOK COMPLETE")
print("="*60)
print(f"\nOutputs saved to:")
print(f"  Models: {Config.MODEL_DIR}")
print(f"  Logs: {Config.LOG_DIR}")
print(f"  TensorBoard: {Config.TENSORBOARD_LOG}")
print(f"\nTo view TensorBoard logs, run:")
print(f"  tensorboard --logdir {Config.TENSORBOARD_LOG}")

---
## Appendix: Hyperparameter Guide
---

### PPO Hyperparameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `learning_rate` | 2.5e-4 | Learning rate. Lower for stability, higher for faster learning |
| `n_steps` | 128 | Steps per environment per update. Higher = more stable, slower |
| `batch_size` | 256 | Minibatch size. Should divide `n_steps * n_envs` evenly |
| `n_epochs` | 4 | Optimization epochs per update |
| `gamma` | 0.99 | Discount factor. Higher = longer-term planning |
| `gae_lambda` | 0.95 | GAE lambda. Trade-off between bias and variance |
| `clip_range` | 0.1 | PPO clipping parameter. Lower = more conservative updates |
| `ent_coef` | 0.01 | Entropy coefficient. Higher = more exploration |
| `vf_coef` | 0.5 | Value function coefficient |
| `max_grad_norm` | 0.5 | Max gradient norm for clipping |

### Environment Settings

| Parameter | Default | Description |
|-----------|---------|-------------|
| `frame_skip` | 4 | Repeat action for N frames. Higher = faster training |
| `frame_stack` | 4 | Stack N frames for temporal information |
| `resize_shape` | (84, 84) | Resize observation. Smaller = faster, but less detail |
| `n_envs` | 8 | Parallel environments. More = faster but uses more memory |

### Curriculum Learning Extensions

For curriculum-based experiments, modify:

1. **Action Space Curriculum**: Start with `RIGHT_ONLY`, then `SIMPLE_MOVEMENT`, then `COMPLEX_MOVEMENT`
2. **Level Curriculum**: Start with World 1, progressively add harder worlds
3. **Reward Curriculum**: Start with survival only, then add movement reward, then full reward

---